In [ ]:
import mne
import numpy as np
from pathlib import Path

DATA_DIR = Path("/home/shivansh/Documents/Programs/Internship_ke_liye_projects/archive/files")
recording_files = sorted(DATA_DIR.glob("S*/S*R08.edf"))

TARGET_SFREQ = 160.0

all_X = []
all_y = []
all_subjects = []

for file_path in recording_files:

    subject = file_path.parent.name
    print(f"\nProcessing {subject}...")

    # 1. Load EEG recording
    raw = mne.io.read_raw_edf(
        file_path,
        preload=True,
        verbose=False
    )

    original_sfreq = raw.info["sfreq"]
    print(f"  Original sampling frequency: {original_sfreq} Hz")

    # 2. Standardize channel names
    raw.rename_channels({
        ch: ch.rstrip(".")
        for ch in raw.ch_names
    })

    # 3. Assign electrode positions
    montage = mne.channels.make_standard_montage(
        "standard_1005"
    )

    raw.set_montage(
        montage,
        match_case=False,
        on_missing="raise"
    )

    # 4. Resample every recording to 160 Hz
    if original_sfreq != TARGET_SFREQ:
        raw.resample(
            sfreq=TARGET_SFREQ,
            npad="auto",
            verbose=False
        )

    print(f"  Resampled frequency: {raw.info['sfreq']} Hz")

    # 5. Apply bandpass filter
    raw.filter(
        l_freq=8.0,
        h_freq=30.0,
        picks="eeg",
        method="fir",
        phase="zero",
        verbose=False
    )

    # 6. Extract events
    events, event_id = mne.events_from_annotations(
        raw,
        verbose=False
    )

    # Check that both motor imagery classes exist
    if "T1" not in event_id or "T2" not in event_id:
        print(f"  Skipping {subject}: missing T1 or T2")
        continue

    motor_event_id = {
        "left_hand": event_id["T1"],
        "right_hand": event_id["T2"]
    }

    # 7. Extract trials
    epochs = mne.Epochs(
        raw,
        events,
        event_id=motor_event_id,
        tmin=0.5,
        tmax=3.5,
        baseline=None,
        preload=True,
        reject_by_annotation=True,
        verbose=False
    )

    # 8. Convert epochs into arrays
    X_subject = epochs.get_data()

    event_codes = epochs.events[:, -1]

    # Explicitly assign labels
    y_subject = np.where(
        event_codes == event_id["T1"],
        0,
        1
    )

    # 9. Store data
    if len(y_subject) == 0:
        print(f"  No usable trials for {subject}")
        continue

    all_X.append(X_subject)
    all_y.append(y_subject)

    participant_id = int(subject[1:])

    all_subjects.extend(
        [participant_id] * len(y_subject)
    )

    print(f"  Extracted trials: {len(y_subject)}")
    print(f"  Trial shape: {X_subject.shape[1:]}")

print("\nPreprocessing complete!")


Processing S001...
  Original sampling frequency: 160.0 Hz
  Resampled frequency: 160.0 Hz
  Extracted trials: 15
  Trial shape: (64, 481)

Processing S002...
  Original sampling frequency: 160.0 Hz
  Resampled frequency: 160.0 Hz
  Extracted trials: 15
  Trial shape: (64, 481)

Processing S003...
  Original sampling frequency: 160.0 Hz
  Resampled frequency: 160.0 Hz
  Extracted trials: 15
  Trial shape: (64, 481)

Processing S004...
  Original sampling frequency: 160.0 Hz
  Resampled frequency: 160.0 Hz
  Extracted trials: 15
  Trial shape: (64, 481)

Processing S005...
  Original sampling frequency: 160.0 Hz
  Resampled frequency: 160.0 Hz
  Extracted trials: 15
  Trial shape: (64, 481)

Processing S006...
  Original sampling frequency: 160.0 Hz
  Resampled frequency: 160.0 Hz
  Extracted trials: 15
  Trial shape: (64, 481)

Processing S007...
  Original sampling frequency: 160.0 Hz
  Resampled frequency: 160.0 Hz
  Extracted trials: 15
  Trial shape: (64, 481)

Processing S008...


In [ ]:
X = np.concatenate(all_X, axis=0)
y = np.concatenate(all_y, axis=0)
subjects = np.asarray(all_subjects)

print("Dataset shape:", X.shape)
print("Labels shape:", y.shape)
print("Subjects shape:", subjects.shape)

print("Unique participants:", len(np.unique(subjects)))
print("Total trials:", len(y))
print("Left-hand trials:", np.sum(y == 0))
print("Right-hand trials:", np.sum(y == 1))

# Verify consistency
assert X.shape[0] == len(y) == len(subjects)
assert X.shape[2] == 481

print("\nAll dimensions are consistent!")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

# Find trials from each class
left_idx = np.where(y == 0)[0][0]
right_idx = np.where(y == 1)[0][0]

# Find the C3 channel
channel_names = [
    ch.rstrip(".")
    for ch in raw.ch_names
]

c3_idx = channel_names.index("C3")

# Time axis: 481 samples at 160 Hz, starting 0.5 seconds after onset
times = 0.5 + np.arange(X.shape[2]) / 160.0

# Convert voltage from volts to microvolts
left_signal = X[left_idx, c3_idx, :] * 1e6
right_signal = X[right_idx, c3_idx, :] * 1e6

plt.figure(figsize=(12, 5))

plt.plot(times, left_signal, label="Left-hand imagery")
plt.plot(times, right_signal, label="Right-hand imagery", alpha=0.8)

plt.xlabel("Time after event onset (seconds)")
plt.ylabel("EEG amplitude (µV)")
plt.title("C3 EEG Signals: Left vs Right Motor Imagery")
plt.legend()
plt.grid(True, alpha=0.3)
plt.show()

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.model_selection import GroupKFold, cross_validate
from sklearn.discriminant_analysis import LinearDiscriminantAnalysis
from sklearn.metrics import make_scorer, balanced_accuracy_score
from mne.decoding import CSP
import numpy as np

# Five-fold participant-independent cross-validation
cv = GroupKFold(n_splits=5)

# CSP + LDA pipeline
model = Pipeline([
                ("csp",CSP(n_components=4, reg=None, log=True, norm_trace=False, cov_est="concat"))
                ,("lda", LinearDiscriminantAnalysis())
                ])

# Evaluate the model
results = cross_validate(
    model, X, y, groups=subjects, cv=cv, scoring={
        "accuracy": "accuracy", "balanced_accuracy": make_scorer(balanced_accuracy_score)},
    return_train_score=True, n_jobs=1
)

# Display results
print("Fold-wise test accuracy:")
print(np.round(results["test_accuracy"], 4))

print("\nFold-wise balanced accuracy:")
print(np.round(results["test_balanced_accuracy"], 4))

print("\nMean test accuracy:",
      round(results["test_accuracy"].mean(), 4))

print("Mean balanced accuracy:",
      round(results["test_balanced_accuracy"].mean(), 4))

print("\nMean training accuracy:",
      round(results["train_accuracy"].mean(), 4))

csp_lda_accuracy = round(results["test_accuracy"].mean(), 4)
csp_lda_balanced_accuracy = round(results["test_balanced_accuracy"].mean(), 4)
csp_lda_train_accuracy = round(results["train_accuracy"].mean(), 4)

In [ ]:
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Reproducibility
np.random.seed(42)
torch.manual_seed(42)

# Convert EEG data to float32 for PyTorch
X_cnn = X.astype(np.float32)
y_cnn = y.astype(np.int64)

print("X shape:", X_cnn.shape)
print("y shape:", y_cnn.shape)
print("Unique labels:", np.unique(y_cnn))
print("Number of participants:", len(np.unique(subjects)))


In [ ]:
import torch
import torch.nn as nn

class EEG_CNN(nn.Module):
    def __init__(self):
        super().__init__()

        self.features = nn.Sequential(
            # Input: (batch, 64, 481)
            nn.Conv1d(
                in_channels=64,
                out_channels=32,
                kernel_size=7,
                padding=3
            ),
            nn.BatchNorm1d(32),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            # Temporal dimension: approximately 240
            nn.Conv1d(
                in_channels=32,
                out_channels=64,
                kernel_size=5,
                padding=2
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),
            nn.MaxPool1d(kernel_size=2),

            # Temporal dimension: approximately 120
            nn.Conv1d(
                in_channels=64,
                out_channels=64,
                kernel_size=3,
                padding=1
            ),
            nn.BatchNorm1d(64),
            nn.ReLU(),

            # Aggregate temporal features
            nn.AdaptiveAvgPool1d(1)
        )

        self.classifier = nn.Sequential(
            nn.Flatten(),
            nn.Dropout(0.4),
            nn.Linear(64, 2)
        )

    def forward(self, x):
        x = self.features(x)
        x = self.classifier(x)
        return x


model = EEG_CNN()

print(model)
print(
    "Trainable parameters:",
    sum(p.numel() for p in model.parameters() if p.requires_grad)
)

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold
from sklearn.metrics import accuracy_score, balanced_accuracy_score

# Reproducibility
torch.manual_seed(42)
np.random.seed(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

# Five-fold participant-wise cross-validation
cv = GroupKFold(n_splits=5)

cnn_test_accuracy = []
cnn_test_balanced_accuracy = []
cnn_train_accuracy = []

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X_cnn, y_cnn, groups=subjects), start=1
):
    print(f"\nFold {fold}/5")

    # Convert data to PyTorch tensors
    X_train = torch.tensor(X_cnn[train_idx], dtype=torch.float32)
    y_train = torch.tensor(y_cnn[train_idx], dtype=torch.long)

    X_test = torch.tensor(X_cnn[test_idx], dtype=torch.float32)
    y_test = torch.tensor(y_cnn[test_idx], dtype=torch.long)

    # Normalize each trial/channel using only that trial's time samples.
    # This avoids estimating normalization statistics from test participants.
    train_mean = X_train.mean(dim=2, keepdim=True)
    train_std = X_train.std(dim=2, keepdim=True).clamp_min(1e-6)
    X_train = (X_train - train_mean) / train_std

    test_mean = X_test.mean(dim=2, keepdim=True)
    test_std = X_test.std(dim=2, keepdim=True).clamp_min(1e-6)
    X_test = (X_test - test_mean) / test_std

    train_dataset = TensorDataset(X_train, y_train)
    train_loader = DataLoader(
        train_dataset, batch_size=32, shuffle=True
    )

    # A fresh model for every fold
    model = EEG_CNN().to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.Adam(
        model.parameters(), lr=0.001, weight_decay=1e-4
    )

    # Training
    epochs = 30

    for epoch in range(epochs):
        model.train()

        for batch_X, batch_y in train_loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

    # Evaluate on unseen participants
    model.eval()

    with torch.no_grad():
        train_outputs = model(X_train.to(device))
        test_outputs = model(X_test.to(device))

        train_predictions = train_outputs.argmax(dim=1).cpu().numpy()
        test_predictions = test_outputs.argmax(dim=1).cpu().numpy()

    train_accuracy = accuracy_score(
        y_train.numpy(), train_predictions
    )
    test_accuracy = accuracy_score(
        y_test.numpy(), test_predictions
    )
    test_balanced_accuracy = balanced_accuracy_score(
        y_test.numpy(), test_predictions
    )

    cnn_train_accuracy.append(train_accuracy)
    cnn_test_accuracy.append(test_accuracy)
    cnn_test_balanced_accuracy.append(test_balanced_accuracy)

    print(f"  Train accuracy:      {train_accuracy:.4f}")
    print(f"  Test accuracy:       {test_accuracy:.4f}")
    print(f"  Balanced accuracy:   {test_balanced_accuracy:.4f}")

print("\n========== CNN RESULTS ==========")
print("Fold-wise test accuracy:", np.round(cnn_test_accuracy, 4))
print("Fold-wise balanced accuracy:",
      np.round(cnn_test_balanced_accuracy, 4))
print(f"Mean test accuracy: {np.mean(cnn_test_accuracy):.4f}")
print(f"Mean balanced accuracy: "
      f"{np.mean(cnn_test_balanced_accuracy):.4f}")
print(f"Mean training accuracy: {np.mean(cnn_train_accuracy):.4f}")

CNN_accuracy = round(np.mean(cnn_test_accuracy), 4)
CNN_balanced_accuracy = round(np.mean(cnn_test_balanced_accuracy), 4)
CNN_trained_accuracy =  round(np.mean(cnn_train_accuracy), 4)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

models = ["CSP + LDA", "CNN"]
accuracy = [csp_lda_accuracy *100, CNN_accuracy*100]
balanced_accuracy = [csp_lda_balanced_accuracy*100, CNN_balanced_accuracy*100]

x = np.arange(len(models))
width = 0.35

fig, ax = plt.subplots(figsize=(8, 5))

bars1 = ax.bar(
    x - width / 2, accuracy, width, label="Test Accuracy"
)
bars2 = ax.bar(
    x + width / 2, balanced_accuracy, width,
    label="Balanced Accuracy"
)

ax.set_ylabel("Score (%)")
ax.set_title("Motor Imagery Classification: Model Comparison")
ax.set_xticks(x)
ax.set_xticklabels(models)
ax.set_ylim(0, 100)
ax.legend()

ax.bar_label(bars1, fmt="%.2f%%", padding=3)
ax.bar_label(bars2, fmt="%.2f%%", padding=3)

plt.tight_layout()
plt.show()

In [ ]:
import pandas as pd

results_table = pd.DataFrame({
    "Model": ["CSP + LDA", "1D CNN"],
    "Mean Test Accuracy (%)": [csp_lda_accuracy *100, CNN_accuracy*100],
    "Mean Balanced Accuracy (%)": [csp_lda_balanced_accuracy*100, CNN_balanced_accuracy*100],
    "Mean Training Accuracy (%)": [csp_lda_train_accuracy*100, CNN_trained_accuracy*100]
})

results_table

In [ ]:
results_table.to_csv(
    "../results/model_comparison.csv",
    index=False
)

print("Results saved successfully.")

In [ ]:
import copy
import numpy as np
import torch
import torch.nn as nn
from torch.utils.data import TensorDataset, DataLoader
from sklearn.model_selection import GroupKFold, GroupShuffleSplit
from sklearn.metrics import accuracy_score, balanced_accuracy_score

def train_with_early_stopping(
    X_train, y_train, groups_train,
    X_val, y_val,
    device,
    max_epochs=60,
    patience=8
):
    # Normalize each trial/channel across its time samples
    X_train = X_train.astype(np.float32).copy()
    X_val = X_val.astype(np.float32).copy()

    train_mean = X_train.mean(axis=2, keepdims=True)
    train_std = X_train.std(axis=2, keepdims=True).clip(min=1e-6)
    X_train = (X_train - train_mean) / train_std

    val_mean = X_val.mean(axis=2, keepdims=True)
    val_std = X_val.std(axis=2, keepdims=True).clip(min=1e-6)
    X_val = (X_val - val_mean) / val_std

    X_train = torch.tensor(X_train, dtype=torch.float32)
    y_train = torch.tensor(y_train, dtype=torch.long)
    X_val = torch.tensor(X_val, dtype=torch.float32)
    y_val = torch.tensor(y_val, dtype=torch.long)

    loader = DataLoader(
        TensorDataset(X_train, y_train),
        batch_size=32,
        shuffle=True
    )

    model = EEG_CNN().to(device)
    criterion = nn.CrossEntropyLoss()

    optimizer = torch.optim.Adam(
        model.parameters(),
        lr=0.001,
        weight_decay=1e-3
    )

    best_val_loss = float("inf")
    best_state = None
    best_epoch = 0
    epochs_without_improvement = 0

    for epoch in range(max_epochs):
        model.train()

        for batch_X, batch_y in loader:
            batch_X = batch_X.to(device)
            batch_y = batch_y.to(device)

            optimizer.zero_grad()
            outputs = model(batch_X)
            loss = criterion(outputs, batch_y)
            loss.backward()
            optimizer.step()

        # Validation: never update weights here
        model.eval()
        with torch.no_grad():
            val_outputs = model(X_val.to(device))
            val_loss = criterion(val_outputs, y_val.to(device)).item()

        if val_loss < best_val_loss:
            best_val_loss = val_loss
            best_state = copy.deepcopy(model.state_dict())
            best_epoch = epoch + 1
            epochs_without_improvement = 0
        else:
            epochs_without_improvement += 1

        if epochs_without_improvement >= patience:
            break

    model.load_state_dict(best_state)
    return model, best_epoch

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
cv = GroupKFold(n_splits=5)

improved_accuracy = []
improved_balanced_accuracy = []
best_epochs = []

for fold, (train_idx, test_idx) in enumerate(
    cv.split(X_cnn, y_cnn, groups=subjects), start=1
):
    print(f"\nFold {fold}/5")

    # Split training participants into train and validation groups
    splitter = GroupShuffleSplit(
        n_splits=1,
        test_size=0.2,
        random_state=42
    )

    train_rel, val_rel = next(
        splitter.split(
            X_cnn[train_idx],
            y_cnn[train_idx],
            groups=subjects[train_idx]
        )
    )

    actual_train_idx = train_idx[train_rel]
    val_idx = train_idx[val_rel]

    model, best_epoch = train_with_early_stopping(
        X_cnn[actual_train_idx],
        y_cnn[actual_train_idx],
        subjects[actual_train_idx],
        X_cnn[val_idx],
        y_cnn[val_idx],
        device
    )

    # Test participants were excluded from training and validation
    X_test = X_cnn[test_idx].astype(np.float32).copy()
    X_test = (X_test - X_test.mean(axis=2, keepdims=True)) / (
        X_test.std(axis=2, keepdims=True).clip(min=1e-6)
    )

    model.eval()
    with torch.no_grad():
        logits = model(
            torch.tensor(X_test, dtype=torch.float32).to(device)
        )
        predictions = logits.argmax(dim=1).cpu().numpy()

    acc = accuracy_score(y_cnn[test_idx], predictions)
    bal_acc = balanced_accuracy_score(y_cnn[test_idx], predictions)

    improved_accuracy.append(acc)
    improved_balanced_accuracy.append(bal_acc)
    best_epochs.append(best_epoch)

    print(f"  Best epoch: {best_epoch}")
    print(f"  Test accuracy: {acc:.4f}")
    print(f"  Balanced accuracy: {bal_acc:.4f}")

print("\n===== IMPROVED CNN RESULTS =====")
print("Fold accuracies:", np.round(improved_accuracy, 4))
print(f"Mean test accuracy: {np.mean(improved_accuracy):.4f}")
print(f"Mean balanced accuracy: {np.mean(improved_balanced_accuracy):.4f}")
print(f"Mean best epoch: {np.mean(best_epochs):.1f}")

In [ ]:
import pandas as pd

final_results = pd.DataFrame({
    "Model": [
        "CSP + LDA",
        "Original CNN",
        "CNN + Early Stopping + Weight Decay"
    ],
    "Mean Test Accuracy (%)": [
        csp_lda_accuracy * 100,
        round(np.mean(cnn_test_accuracy) * 100, 2),
        round(np.mean(improved_accuracy) * 100, 2)
    ],
    "Mean Balanced Accuracy (%)": [
        csp_lda_balanced_accuracy * 100,
        round(np.mean(cnn_test_balanced_accuracy) * 100, 2),
        round(np.mean(improved_balanced_accuracy) * 100, 2)
    ]
})

display(final_results)

final_results.to_csv(
    "../results/final_model_comparison.csv",
    index=False
)

print("Final comparison saved.")